# Kinetic-Electron Analytical Benchmarks

Tests for gyaradax with **kinetic electrons** (`adiabatic_electrons=False`), focusing on physics that requires the parallel electron response. These are independent of the EM (`A_\parallel`) tests in `em_analytical.ipynb` — here all runs are at $\beta = 0$.

| § | Test | Physics | Reference |
|---|------|---------|-----------|
| 1 | **Drift wave** | $\omega = \omega_{*e} = k_y \rho_i \, R/L_n$ | analytic |
| 2 | **Ion-acoustic wave** | $\omega = k_\parallel c_s$ with $c_s = \sqrt{T_e/m_i}$ | analytic |
| 3 | **TEM linear scan** | $\gamma(k_y)$ at TEM-favorable params ($R/L_{Ti}=0$, $R/L_{Te} \gg 0$) | GKW paper Fig 4 left, Peeters PoP 2005 |
| 4 | **Cyclone CBC: kinetic vs adiabatic** | TEM enhancement of CBC | Dimits PoP 2000, Lapillonne PoP 2009 |

Sections 1–2 are pure-analytical (closed-form dispersion). Section 3 is a numerical benchmark vs the published GKW reference. Section 4 contrasts kinetic-electron and adiabatic-electron treatment of the same Cyclone Base Case.

All long runs cache to `data/kinetic_analytical/*.npz`; delete a cache file to force a rerun.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), ".."))
os.environ["CUDA_VISIBLE_DEVICES"] = "7"
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"

In [ ]:
import time as _time
import numpy as np
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
from dataclasses import replace

from gyaradax.geometry import compute_geometry
from gyaradax.params import GKParams
from gyaradax.simulate import gk_run
from gyaradax.solver import init_f, default_state, linear_precompute
from gyaradax.plot_utils import JAX_COLORS

CACHE_DIR = "data/kinetic_analytical"
os.makedirs(CACHE_DIR, exist_ok=True)
os.makedirs("figs", exist_ok=True)

def _cache_load(name):
    p = os.path.join(CACHE_DIR, name)
    return dict(np.load(p, allow_pickle=False)) if os.path.exists(p) else None

def _cache_save(name, **arrays):
    p = os.path.join(CACHE_DIR, name)
    np.savez(p, **arrays)
    print(f"  saved {p}")

## 1. Drift wave dispersion $\omega = \omega_{*e}$

The most basic linear test of kinetic electrons. With finite density gradient $R/L_n > 0$ and zero temperature gradients ($R/L_{Ti} = R/L_{Te} = 0$), the system supports the **electron drift wave**: a low-frequency wave propagating in the electron diamagnetic direction with

$$\omega \;=\; \omega_{*e} \;=\; \frac{k_\theta T_e}{e B L_n}\ ,$$

which in GKW-normalised units reduces to $\omega = k_\theta \rho_i \cdot (R/L_n)$ for $T_e = T_i$ and the kinetic deuterium as reference. The wave is marginal at the long-wavelength FLR-free limit (drift waves require FLR + kinetic ion response to grow); for kinetic ions + Boltzmann-like response and small FLR the **real frequency** still equals $\omega_{*e}$.

We scan $k_\theta \rho_i$ at fixed $R/L_n = 1$ and zero T gradients, kinetic electrons + ions. Measure $\omega$ from the phase evolution of $\phi$ in the late stage.

In [ ]:
def run_drift_wave(krho, rln=1.0, n_windows=200, navg=20, dt=0.01,
                   q=1.4, eps=0.18, shat=0.0, me_ratio=0.01):
    """Drift wave linear test: finite R/L_n, zero R/L_T, kinetic e-.

    Measure complex omega from late-time phase + amplitude of phi(t).
    Heavy electrons (m_e/m_i = 0.01) to relax the streaming CFL while
    keeping kinetic e- physics intact (drift wave is parametrically
    independent of the mass ratio in the small-omega/k_par v_th_e limit).
    """
    geom = compute_geometry(
        q=q, shat=shat, eps=eps,
        ns=32, nvpar=32, nmu=8, vpar_max=3.0,
        nkx=1, nky=1, nperiod=2,
        kxmax=krho, krhomax=krho, signB=1.0, Rref=1.0,
        geom_type='s-alpha',
    )
    params = GKParams(
        dt=dt, naverage=navg, non_linear=False, adaptive_dt=True,
        adiabatic_electrons=False,
        disp_par=1.0, disp_vp=0.1, disp_x=0.0, disp_y=0.0,
        finit='cosine2', amp_init=1e-3,
        mas=jnp.array([1.0, me_ratio]), tmp=jnp.array([1.0, 1.0]),
        de=jnp.array([1.0, 1.0]), signz=jnp.array([1.0, -1.0]),
        vthrat=jnp.sqrt(jnp.array([1.0, 1.0]) / jnp.array([1.0, me_ratio])),
        rlt=jnp.array([0.0, 0.0]), rln=jnp.array([float(rln), float(rln)]),
        dgrid=1.0, tgrid=1.0,
        sgr_dist=float(geom['sgr_dist']),
        dvp=float(geom['dvp']),
        kxmax=float(np.max(np.abs(np.asarray(geom['kxrh'])))) or 1.0,
        kymax=float(np.max(np.asarray(geom['krho']))) or 1.0,
        nlapar=False, nlbpar=False, beta=0.0,
        drive_scale=1.0, idisp=2, cfl_safety=0.9,
        mixed_precision=False, backend='jax',
    )
    df = init_f(geom, finit='cosine2', amp_init_real=params.amp_init, n_species=2)
    pre = linear_precompute(geom, params)
    state = default_state(nky=len(geom['krho']))
    phi_phases, times = [], []
    for _ in range(n_windows):
        df, phi, _, state = gk_run(df, geom, params, state, navg, pre=pre)
        ns = phi.shape[0]
        c = phi[ns // 2, 0, 0]
        phi_phases.append(float(jnp.angle(c)))
        times.append(float(state.time))
    ph = np.unwrap(np.asarray(phi_phases))
    tt = np.asarray(times)
    mask = tt > tt[-1] * 0.5
    omega = float(np.polyfit(tt[mask], ph[mask], 1)[0]) if mask.sum() >= 2 else np.nan
    gamma = float(np.asarray(state.last_growth_rate).max())
    return gamma, omega

In [ ]:
_cached = _cache_load("drift_wave.npz")
if _cached is not None:
    krhos_dw = _cached["krho"]
    omegas_dw = _cached["omega"]
    gammas_dw = _cached["gamma"]
    rln_dw = float(_cached["rln"])
    print(f"loaded drift-wave scan ({len(krhos_dw)} pts, R/L_n={rln_dw})")
else:
    rln_dw = 1.0
    krhos_dw = np.array([0.05, 0.1, 0.15, 0.2, 0.3, 0.4, 0.5])
    _w, _g = [], []
    for kr in krhos_dw:
        try:
            g, w = run_drift_wave(float(kr), rln=rln_dw)
        except Exception as e:
            print(f"krho={kr:.3f}  FAILED: {e}")
            g, w = np.nan, np.nan
        _w.append(w); _g.append(g)
        print(f"krho={kr:.3f}  ω={w:+.4f}  ω_*e={kr*rln_dw:+.4f}  γ={g:+.4f}")
    omegas_dw = np.array(_w)
    gammas_dw = np.array(_g)
    _cache_save("drift_wave.npz", krho=krhos_dw, omega=omegas_dw, gamma=gammas_dw,
                rln=np.array(rln_dw))

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(4.0, 3.0))
ax.plot(krhos_dw, omegas_dw, 'o-', color=JAX_COLORS['blue'], ms=6, lw=1.2,
        label='gyaradax')
ax.plot(krhos_dw, krhos_dw * rln_dw, '--k', lw=1.2,
        label=rf'$\omega_{{*e}} = k_\theta\rho_i \cdot R/L_n$ ($R/L_n={rln_dw}$)')
ax.set_xlabel(r'$k_\theta \rho_i$', fontsize=13)
ax.set_ylabel(r'$\omega$  $[v_{\rm th,i}/R]$', fontsize=13)
ax.legend(frameon=False, fontsize=10)
ax.tick_params(labelsize=10)
ax.grid(True, alpha=0.4)
ax.set_title(r'Drift wave dispersion $\omega = \omega_{*e}$', fontsize=13)
fig.tight_layout()
fig.savefig('figs/kin_drift_wave.pdf')
plt.show()

## 2. Ion-acoustic wave dispersion

With kinetic electrons, finite $T_e$, **no drives** ($R/L_T = R/L_n = 0$), and $\beta = 0$, the only linear mode is the ion-acoustic wave (slow wave) with dispersion

$$\omega \;=\; k_\parallel c_s , \qquad c_s = \sqrt{T_e / m_i}.$$

In GKW units with $T_i = T_e = 1$ and the kinetic ion as the reference species, $c_s = v_{\rm th,i} \cdot \sqrt{1/2} \cdot \sqrt{T_e/T_i} = 1/\sqrt{2}$ (the factor $\sqrt{2}$ is the GKW $v_{\rm th} = \sqrt{2T/m}$ definition).

We initialise an $A_\parallel = 0$ perturbation at a single $k_\theta$ with non-zero parallel wavenumber set by the flux-tube boundary conditions, and measure $\omega/k_\parallel$. This is analogous to the shear Alfvén test in `em_analytical.ipynb` but at $\beta = 0$ (no magnetic field perturbation).

In [ ]:
def run_ion_acoustic(krho, n_windows=200, navg=20, dt=0.005,
                    q=2.0, eps=0.166, shat=1.0, nperiod=4, me_ratio=0.01):
    """Ion-acoustic wave: drives off, kinetic e-, beta=0.

    Measure omega from phi(t); compare to k_parallel * c_s with c_s = v_th/sqrt(2)
    (GKW normalisation: v_th = sqrt(2 T/m)).
    """
    geom = compute_geometry(
        q=q, shat=shat, eps=eps,
        ns=64, nvpar=32, nmu=8, vpar_max=3.0,
        nkx=1, nky=1, nperiod=nperiod,
        kxmax=krho, krhomax=krho, signB=1.0, Rref=1.0,
        geom_type='s-alpha',
    )
    params = GKParams(
        dt=dt, naverage=navg, non_linear=False, adaptive_dt=True,
        adiabatic_electrons=False,
        disp_par=1.0, disp_vp=0.05, disp_x=0.0, disp_y=0.0,
        finit='cosine2', amp_init=1e-3,
        mas=jnp.array([1.0, me_ratio]), tmp=jnp.array([1.0, 1.0]),
        de=jnp.array([1.0, 1.0]), signz=jnp.array([1.0, -1.0]),
        vthrat=jnp.sqrt(jnp.array([1.0, 1.0]) / jnp.array([1.0, me_ratio])),
        rlt=jnp.array([0.0, 0.0]), rln=jnp.array([0.0, 0.0]),
        dgrid=1.0, tgrid=1.0,
        sgr_dist=float(geom['sgr_dist']),
        dvp=float(geom['dvp']),
        kxmax=float(np.max(np.abs(np.asarray(geom['kxrh'])))) or 1.0,
        kymax=float(np.max(np.asarray(geom['krho']))) or 1.0,
        nlapar=False, nlbpar=False, beta=0.0,
        drive_scale=1.0, idisp=2, cfl_safety=0.9,
        mixed_precision=False, backend='jax',
    )
    df = init_f(geom, finit='cosine2', amp_init_real=params.amp_init, n_species=2)
    pre = linear_precompute(geom, params)
    state = default_state(nky=len(geom['krho']))
    phi_phases, times = [], []
    for _ in range(n_windows):
        df, phi, _, state = gk_run(df, geom, params, state, navg, pre=pre)
        ns = phi.shape[0]
        c = phi[ns // 2, 0, 0]
        phi_phases.append(float(jnp.angle(c)))
        times.append(float(state.time))
    ph = np.unwrap(np.asarray(phi_phases))
    tt = np.asarray(times)
    mask = tt > tt[-1] * 0.5
    omega = float(np.polyfit(tt[mask], ph[mask], 1)[0]) if mask.sum() >= 2 else np.nan
    return omega

In [ ]:
_cached = _cache_load("ion_acoustic.npz")
if _cached is not None:
    krhos_iaw = _cached["krho"]
    omegas_iaw = _cached["omega"]
    print(f"loaded ion-acoustic scan ({len(krhos_iaw)} pts)")
else:
    krhos_iaw = np.array([0.1, 0.2, 0.3, 0.5, 0.7])
    _w = []
    for kr in krhos_iaw:
        try:
            w = run_ion_acoustic(float(kr))
        except Exception as e:
            print(f"krho={kr:.3f}  FAILED: {e}")
            w = np.nan
        _w.append(w)
        print(f"krho={kr:.3f}  |ω|={abs(w):.4f}")
    omegas_iaw = np.array(_w)
    _cache_save("ion_acoustic.npz", krho=krhos_iaw, omega=omegas_iaw)

In [ ]:
# eff k_par from omega: assuming omega = k_par * c_s with c_s = 1/sqrt(2)
c_s = 1.0 / np.sqrt(2.0)
kpar_eff = np.abs(omegas_iaw) / c_s
fig, ax = plt.subplots(1, 1, figsize=(4.0, 3.0))
ax.plot(krhos_iaw, np.abs(omegas_iaw), 'o-', color=JAX_COLORS['blue'], ms=6,
        label='gyaradax')
k_const = np.median(kpar_eff)
ax.axhline(k_const * c_s, color='k', ls='--', lw=1.0,
           label=rf'$k_\parallel c_s$, $k_\parallel={k_const:.3f}$ (median)')
ax.set_xlabel(r'$k_\theta \rho_i$', fontsize=13)
ax.set_ylabel(r'$|\omega|$  $[v_{\rm th,i}/R]$', fontsize=13)
ax.legend(frameon=False, fontsize=10)
ax.tick_params(labelsize=10)
ax.grid(True, alpha=0.4)
ax.set_title(r'Ion-acoustic wave: $\omega = k_\parallel c_s$', fontsize=13)
fig.tight_layout()
fig.savefig('figs/kin_ion_acoustic.pdf')
plt.show()

## 3. TEM linear scan (TEM-favorable parameters)

The trapped-electron-mode (TEM) is destabilised when the **electron** temperature/density gradients dominate over the ion drive. We pick a canonical TEM-favorable set (no ITG drive, strong electron drive):

- $q = 1.4$, $\hat s = 0.8$, $\varepsilon = 0.18$ (ASDEX-like)
- $R/L_{Ti} = 0$, $R/L_{Te} = 6.9$, $R/L_n = 2.2$
- kinetic deuterium + electrons, $\beta = 0$, single $k_\theta$ scan

This is the standard kinetic-electron benchmark used in cross-code comparison papers (e.g. Falchetto PPCF 2008, Lapillonne PoP 2009). Pure TEM modes propagate in the **electron diamagnetic direction** ($\omega > 0$ in GKW convention).

Heavy electrons ($m_e/m_i = 0.01$) are used to make the trapped-electron physics tractable at reasonable dt; the qualitative TEM behaviour persists down to the physical mass ratio.

In [ ]:
def run_tem_linear(krho, n_windows=200, navg=20, dt=0.01,
                   q=1.4, eps=0.18, shat=0.8,
                   rlte=6.9, rln=2.2, me_ratio=0.01):
    """Linear TEM growth rate: R/L_Ti=0, strong electron drive, kinetic e-."""
    geom = compute_geometry(
        q=q, shat=shat, eps=eps,
        ns=32, nvpar=32, nmu=16, vpar_max=3.0,
        nkx=1, nky=1, nperiod=2,
        kxmax=krho, krhomax=krho, signB=1.0, Rref=1.0,
        geom_type='s-alpha',
    )
    params = GKParams(
        dt=dt, naverage=navg, non_linear=False, adaptive_dt=True,
        adiabatic_electrons=False,
        disp_par=1.0, disp_vp=0.1, disp_x=0.0, disp_y=0.0,
        finit='cosine2', amp_init=1e-3,
        mas=jnp.array([1.0, me_ratio]), tmp=jnp.array([1.0, 1.0]),
        de=jnp.array([1.0, 1.0]), signz=jnp.array([1.0, -1.0]),
        vthrat=jnp.sqrt(jnp.array([1.0, 1.0]) / jnp.array([1.0, me_ratio])),
        rlt=jnp.array([0.0, float(rlte)]),
        rln=jnp.array([float(rln), float(rln)]),
        dgrid=1.0, tgrid=1.0,
        sgr_dist=float(geom['sgr_dist']),
        dvp=float(geom['dvp']),
        kxmax=float(np.max(np.abs(np.asarray(geom['kxrh'])))) or 1.0,
        kymax=float(np.max(np.asarray(geom['krho']))) or 1.0,
        nlapar=False, nlbpar=False, beta=0.0,
        drive_scale=1.0, idisp=2, cfl_safety=0.9,
        mixed_precision=False, backend='jax',
    )
    df = init_f(geom, finit='cosine2', amp_init_real=params.amp_init, n_species=2)
    pre = linear_precompute(geom, params)
    state = default_state(nky=len(geom['krho']))
    phi_phases, times = [], []
    for _ in range(n_windows):
        df, phi, _, state = gk_run(df, geom, params, state, navg, pre=pre)
        ns = phi.shape[0]
        c = phi[ns // 2, 0, 0]
        phi_phases.append(float(jnp.angle(c)))
        times.append(float(state.time))
    ph = np.unwrap(np.asarray(phi_phases))
    tt = np.asarray(times)
    mask = tt > tt[-1] * 0.5
    omega = float(np.polyfit(tt[mask], ph[mask], 1)[0]) if mask.sum() >= 2 else np.nan
    gamma = float(np.asarray(state.last_growth_rate).max())
    return gamma, omega

In [ ]:
_cached = _cache_load("tem_linear.npz")
if _cached is not None:
    krhos_tem = _cached["krho"]
    gammas_tem = _cached["gamma"]
    omegas_tem = _cached["omega"]
    print(f"loaded TEM scan ({len(krhos_tem)} pts)")
else:
    krhos_tem = np.array([0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8])
    _g, _w = [], []
    for kr in krhos_tem:
        try:
            g, w = run_tem_linear(float(kr))
        except Exception as e:
            print(f"krho={kr:.3f}  FAILED: {e}")
            g, w = np.nan, np.nan
        _g.append(g); _w.append(w)
        print(f"krho={kr:.3f}  γ={g:+.4f}  ω={w:+.4f}")
    gammas_tem = np.array(_g)
    omegas_tem = np.array(_w)
    _cache_save("tem_linear.npz", krho=krhos_tem, gamma=gammas_tem, omega=omegas_tem)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(7.0, 2.6))
ax = axes[0]
ax.plot(krhos_tem, gammas_tem, 'o-', color=JAX_COLORS['blue'], ms=6, lw=1.2,
        label='gyaradax')
ax.axhline(0, color='0.6', ls=':', lw=0.6)
ax.set_xlabel(r'$k_\theta \rho_i$', fontsize=13)
ax.set_ylabel(r'$\gamma$  $[v_{\rm th,i}/R]$', fontsize=13)
ax.set_title('(a) TEM growth rate', fontsize=13)
ax.legend(frameon=False, fontsize=10)
ax.tick_params(labelsize=10)
ax.grid(True, alpha=0.4)

ax = axes[1]
ax.plot(krhos_tem, omegas_tem, 'o-', color=JAX_COLORS['orange'], ms=6, lw=1.2,
        label='gyaradax')
ax.axhline(0, color='0.6', ls=':', lw=0.6)
ax.set_xlabel(r'$k_\theta \rho_i$', fontsize=13)
ax.set_ylabel(r'$\omega$  $[v_{\rm th,i}/R]$', fontsize=13)
ax.set_title('(b) TEM real frequency (electron direction)', fontsize=11)
ax.legend(frameon=False, fontsize=10)
ax.tick_params(labelsize=10)
ax.grid(True, alpha=0.4)

fig.suptitle(r'TEM benchmark: $q=1.4$, $\hat s=0.8$, $\varepsilon=0.18$, $R/L_{Te}=6.9$, $R/L_n=2.2$, $R/L_{Ti}=0$',
             fontsize=10, y=1.02)
fig.tight_layout()
fig.savefig('figs/kin_tem_linear.pdf')
plt.show()

## 4. Cyclone CBC: kinetic vs adiabatic electrons

Standard Cyclone Base Case parameters (Dimits PoP 2000):

- $q = 1.4$, $\hat s = 0.8$, $\varepsilon = 0.18$
- $R/L_{Ti} = R/L_{Te} = 6.9$, $R/L_n = 2.2$
- $T_i = T_e$

Adiabatic electrons give pure ITG; kinetic electrons add trapped-electron destabilisation, typically increasing $\gamma$ by 30–50% across the unstable $k_\theta$ range and shifting the spectrum peak slightly upward in $k_\theta\rho_i$. This is a sanity test that the kinetic-electron infrastructure is wired correctly.

In [ ]:
def run_cbc(krho, adiabatic, n_windows=200, navg=20, dt=0.01,
            q=1.4, eps=0.18, shat=0.8, me_ratio=0.01):
    """Cyclone CBC linear growth rate, adiabatic or kinetic electrons."""
    geom = compute_geometry(
        q=q, shat=shat, eps=eps,
        ns=32, nvpar=32, nmu=8, vpar_max=3.0,
        nkx=1, nky=1, nperiod=2,
        kxmax=krho, krhomax=krho, signB=1.0, Rref=1.0,
        geom_type='s-alpha',
    )
    rlt = 6.9; rln = 2.2
    if adiabatic:
        params = GKParams(
            dt=dt, naverage=navg, non_linear=False, adaptive_dt=True,
            adiabatic_electrons=True,
            disp_par=1.0, disp_vp=0.1, disp_x=0.0, disp_y=0.0,
            finit='cosine2', amp_init=1e-3,
            mas=jnp.array([1.0]), tmp=jnp.array([1.0]),
            de=jnp.array([1.0]), signz=jnp.array([1.0]),
            vthrat=jnp.array([1.0]),
            rlt=jnp.array([rlt]), rln=jnp.array([rln]),
            dgrid=1.0, tgrid=1.0,
            sgr_dist=float(geom['sgr_dist']),
            dvp=float(geom['dvp']),
            kxmax=float(np.max(np.abs(np.asarray(geom['kxrh'])))) or 1.0,
            kymax=float(np.max(np.asarray(geom['krho']))) or 1.0,
            nlapar=False, nlbpar=False, beta=0.0,
            drive_scale=1.0, idisp=2, cfl_safety=0.9,
            mixed_precision=False, backend='jax',
        )
        nsp = 1
    else:
        params = GKParams(
            dt=dt, naverage=navg, non_linear=False, adaptive_dt=True,
            adiabatic_electrons=False,
            disp_par=1.0, disp_vp=0.1, disp_x=0.0, disp_y=0.0,
            finit='cosine2', amp_init=1e-3,
            mas=jnp.array([1.0, me_ratio]), tmp=jnp.array([1.0, 1.0]),
            de=jnp.array([1.0, 1.0]), signz=jnp.array([1.0, -1.0]),
            vthrat=jnp.sqrt(jnp.array([1.0, 1.0]) / jnp.array([1.0, me_ratio])),
            rlt=jnp.array([rlt, rlt]), rln=jnp.array([rln, rln]),
            dgrid=1.0, tgrid=1.0,
            sgr_dist=float(geom['sgr_dist']),
            dvp=float(geom['dvp']),
            kxmax=float(np.max(np.abs(np.asarray(geom['kxrh'])))) or 1.0,
            kymax=float(np.max(np.asarray(geom['krho']))) or 1.0,
            nlapar=False, nlbpar=False, beta=0.0,
            drive_scale=1.0, idisp=2, cfl_safety=0.9,
            mixed_precision=False, backend='jax',
        )
        nsp = 2
    df = init_f(geom, finit='cosine2', amp_init_real=params.amp_init, n_species=nsp)
    pre = linear_precompute(geom, params)
    state = default_state(nky=len(geom['krho']))
    phi_phases, times = [], []
    for _ in range(n_windows):
        df, phi, _, state = gk_run(df, geom, params, state, navg, pre=pre)
        ns = phi.shape[0]
        c = phi[ns // 2, 0, 0]
        phi_phases.append(float(jnp.angle(c)))
        times.append(float(state.time))
    ph = np.unwrap(np.asarray(phi_phases))
    tt = np.asarray(times)
    mask = tt > tt[-1] * 0.5
    omega = float(np.polyfit(tt[mask], ph[mask], 1)[0]) if mask.sum() >= 2 else np.nan
    gamma = float(np.asarray(state.last_growth_rate).max())
    return gamma, omega

In [ ]:
_cached = _cache_load("cbc_kinetic_vs_adiab.npz")
if _cached is not None:
    krhos_cbc = _cached["krho"]
    g_adiab = _cached["gamma_adiab"]
    w_adiab = _cached["omega_adiab"]
    g_kinetic = _cached["gamma_kinetic"]
    w_kinetic = _cached["omega_kinetic"]
    print(f"loaded CBC scan ({len(krhos_cbc)} pts)")
else:
    krhos_cbc = np.array([0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7])
    g_a, w_a, g_k, w_k = [], [], [], []
    for kr in krhos_cbc:
        try:
            ga, wa = run_cbc(float(kr), adiabatic=True)
        except Exception as e:
            print(f"adiab krho={kr:.3f}  FAILED: {e}")
            ga, wa = np.nan, np.nan
        try:
            gk, wk = run_cbc(float(kr), adiabatic=False)
        except Exception as e:
            print(f"kinetic krho={kr:.3f}  FAILED: {e}")
            gk, wk = np.nan, np.nan
        g_a.append(ga); w_a.append(wa); g_k.append(gk); w_k.append(wk)
        print(f"krho={kr:.3f}  adiab γ={ga:+.4f}  kinetic γ={gk:+.4f}  (ratio {gk/ga if ga>0 else float('nan'):.2f})")
    g_adiab = np.array(g_a); w_adiab = np.array(w_a)
    g_kinetic = np.array(g_k); w_kinetic = np.array(w_k)
    _cache_save("cbc_kinetic_vs_adiab.npz", krho=krhos_cbc,
                gamma_adiab=g_adiab, omega_adiab=w_adiab,
                gamma_kinetic=g_kinetic, omega_kinetic=w_kinetic)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(7.5, 2.8))
ax = axes[0]
ax.plot(krhos_cbc, g_adiab, 'o-', color=JAX_COLORS['blue'], ms=6, lw=1.2,
        label='adiabatic e-')
ax.plot(krhos_cbc, g_kinetic, 's-', color=JAX_COLORS['orange'], ms=6, lw=1.2,
        label='kinetic e-')
ax.axhline(0, color='0.6', ls=':', lw=0.6)
ax.set_xlabel(r'$k_\theta \rho_i$', fontsize=13)
ax.set_ylabel(r'$\gamma$  $[v_{\rm th,i}/R]$', fontsize=13)
ax.set_title('(a) CBC linear growth rate', fontsize=13)
ax.legend(frameon=False, fontsize=10)
ax.tick_params(labelsize=10)
ax.grid(True, alpha=0.4)

ax = axes[1]
ax.plot(krhos_cbc, w_adiab, 'o-', color=JAX_COLORS['blue'], ms=6, lw=1.2,
        label='adiabatic e-')
ax.plot(krhos_cbc, w_kinetic, 's-', color=JAX_COLORS['orange'], ms=6, lw=1.2,
        label='kinetic e-')
ax.axhline(0, color='0.6', ls=':', lw=0.6)
ax.set_xlabel(r'$k_\theta \rho_i$', fontsize=13)
ax.set_ylabel(r'$\omega$  $[v_{\rm th,i}/R]$', fontsize=13)
ax.set_title('(b) real frequency (ion direction = $\\omega < 0$)', fontsize=11)
ax.legend(frameon=False, fontsize=10)
ax.tick_params(labelsize=10)
ax.grid(True, alpha=0.4)

fig.tight_layout()
fig.savefig('figs/kin_cbc_kinetic_vs_adiab.pdf')
plt.show()